In [2]:
from cobolt.utils import SingleData, MultiomicDataset
from cobolt.model import Cobolt
import os
import gc
import numpy as np
import pandas as pd
import scipy.sparse as sps
import scanpy as sc
import h5py
import scipy.io as sio
import warnings
# from muon import atac as ac
import seaborn as sns

from os.path import join
import matplotlib.pyplot as plt

/home/yanxh/anaconda3/envs/multi-omics/lib/python3.8/site-packages/tqdm-4.65.0-py3.8.egg/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/yanxh/anaconda3/envs/multi-omics/lib/python3.8/site-packages/umap_learn-0.5.3-py3.8.egg/umap/distances.py:1063: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/home/yanxh/anaconda3/envs/multi-omics/lib/python3.8/site-packages/umap_learn-0.5.3-py3.8.egg/umap/distances.py:1071: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' 

In [3]:
def set_col2cat(ad, cols=[]):
    for col in cols:
        ad.obs[col] = ad.obs[col].astype('category')

def unify_colors(queries, color_key, ref_color_dict):
    for q in queries:
        q.obs[color_key] = q.obs[color_key].astype('category')
        q.uns[f'{color_key}_colors'] = [ref_color_dict[_] for _ in q.obs[color_key].cat.categories]
    return queries

def get_umap(ad, use_reps=[]):
    for use_rep in use_reps:
        umap_add_key = f'{use_rep}_umap'
        sc.pp.neighbors(ad, use_rep=use_rep, n_neighbors=15)
        sc.tl.umap(ad)
        ad.obsm[umap_add_key] = ad.obsm['X_umap']
    return ad

def wrap_warn_plot(adata, basis, color, **kwargs):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=UserWarning)
        sc.pl.embedding(adata, basis=basis, color=color, **kwargs)

def wrap_warn_comp_plot(ads, basis, colors, figw=5, figh=4, **kwargs):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for i, color in enumerate(colors):
            _, axes = plt.subplots(1, len(ads), figsize=(figw*len(ads), figh))
            for j, ad in enumerate(ads):
                sc.pl.embedding(ad, basis=basis, color=color, ax=axes[j], show=False, **kwargs)
            plt.show()

In [4]:
def split_ob(ads, ad_ref, ob='obs', key='emb2'):
    len_ads = [_.n_obs for _ in ads]
    if ob=='obsm':
        split_obsms = np.split(ad_ref.obsm[key], np.cumsum(len_ads[:-1]))
        for ad, v in zip(ads, split_obsms):
            ad.obsm[key] = v
    else:
        split_obs = np.split(ad_ref.obs[key].to_list(), np.cumsum(len_ads[:-1]))
        for ad, v in zip(ads, split_obs):
            ad.obs[key] = v
    
def subset_ad(ad, subset_index):
    ad = ad[subset_index].copy()
    return ad

def set_spatial(ad):
    ad.obsm['spatial'] = ad.obs[['array_row', 'array_col']].values
    ad.obsm['spatial'] = ad.obsm['spatial'][:, ::-1]
    ad.obsm['spatial'][:, 1] = -1 * ad.obsm['spatial'][:, 1]
    return ad

def reorder(ad1, ad2):
    shared_barcodes = ad1.obs_names.intersection(ad2.obs_names)
    ad1 = ad1[shared_barcodes].copy()
    ad2 = ad2[shared_barcodes].copy()
    return ad1, ad2

def load_peak_expr(_dir):
    data = sio.mmread(join(_dir, 'data.mtx'))
    cname = pd.read_csv(join(_dir, 'barcode.csv'), index_col=0)['x'].to_list()
    feat = pd.read_csv(join(_dir, 'feat.csv'), index_col=0)['x'].to_list()
    ad = sc.AnnData(sps.csr_matrix(data.T))
    ad.obs_names = cname
    ad.var_names = feat
    return ad

In [5]:
data_dir = '../../../data/processed/Misar-Stereo'
out_dir = './outputs/Misar-Stereo'
os.makedirs(out_dir, exist_ok=True)

In [6]:
ad_mis13_rna = sc.read_h5ad(join(data_dir, 'Misar-E13/ad_rna.h5ad'))
ad_mis13_atac = sc.read_h5ad(join(data_dir, 'Misar-E13/ad_atac.h5ad'))

ad_mis15_rna = sc.read_h5ad(join(data_dir, 'Misar-E15/ad_rna.h5ad'))
ad_mis15_atac = sc.read_h5ad(join(data_dir, 'Misar-E15/ad_atac.h5ad'))

ad_mis18_rna = sc.read_h5ad(join(data_dir, 'Misar-E18/ad_rna.h5ad'))
ad_mis18_atac = sc.read_h5ad(join(data_dir, 'Misar-E18/ad_atac.h5ad'))

ad_ste12_rna = sc.read_h5ad(join(data_dir, 'Stereo-E12/ad_rna.h5ad'))
ad_ste14_rna = sc.read_h5ad(join(data_dir, 'Stereo-E14/ad_rna.h5ad')) 
ad_ste16_rna = sc.read_h5ad(join(data_dir, 'Stereo-E16/ad_rna.h5ad')) 

In [7]:
for ad in [ad_mis13_rna, ad_mis15_rna, ad_ste12_rna]:
    print(ad.shape)

(1777, 25740)
(1949, 25740)
(9556, 25740)


In [9]:
ad_rna_all = sc.concat([ad_mis13_rna, ad_mis15_rna, ad_mis18_rna, ad_ste12_rna, ad_ste14_rna, ad_ste16_rna])

sc.pp.highly_variable_genes(ad_rna_all, flavor='seurat_v3', n_top_genes=10000, batch_key='Sample')
hvg_names = ad_rna_all.var.query('highly_variable').index.to_numpy()

ad_atac_all = sc.concat([ad_mis13_atac, ad_mis15_atac, ad_mis18_atac])

# ac.pp.tfidf(ad_atac_all, scale_factor=1e4)
sc.pp.highly_variable_genes(ad_atac_all, flavor='seurat_v3', n_top_genes=100000, batch_key='Sample')
hvp_names = ad_atac_all.var.query('highly_variable').index.to_numpy()

<frozen importlib._bootstrap>:219: RuntimeWarning: numpy.ufunc size changed, may indicate binary incompatibility. Expected 216 from C header, got 232 from PyObject


In [10]:
mis13_rna_count = sps.csr_matrix(ad_mis13_rna[:, hvg_names].X.astype(np.float32))
mis13_atac_count = sps.csr_matrix(ad_mis13_atac[:, hvp_names].X.astype(np.float32))

mis15_rna_count = sps.csr_matrix(ad_mis15_rna[:, hvg_names].X.astype(np.float32))
mis15_atac_count = sps.csr_matrix(ad_mis15_atac[:, hvp_names].X.astype(np.float32))

mis18_rna_count = sps.csr_matrix(ad_mis18_rna[:, hvg_names].X.astype(np.float32))
mis18_atac_count = sps.csr_matrix(ad_mis18_atac[:, hvp_names].X.astype(np.float32))

ste12_rna_count = sps.csr_matrix(ad_ste12_rna[:, hvg_names].X.astype(np.float32))
ste14_rna_count = sps.csr_matrix(ad_ste14_rna[:, hvg_names].X.astype(np.float32))
ste16_rna_count = sps.csr_matrix(ad_ste16_rna[:, hvg_names].X.astype(np.float32))

mis13_barc = ad_mis13_rna.obs_names.to_numpy()
mis15_barc = ad_mis15_rna.obs_names.to_numpy()
mis18_barc = ad_mis18_rna.obs_names.to_numpy()
ste12_barc = ad_ste12_rna.obs_names.to_numpy()
ste14_barc = ad_ste14_rna.obs_names.to_numpy()
ste16_barc = ad_ste16_rna.obs_names.to_numpy()

rna_feature = hvg_names
atac_feature = hvp_names

In [11]:
mis13_rna = SingleData("GeneExpr", "Misar-E13", rna_feature, mis13_rna_count, mis13_barc)
mis13_atac = SingleData("ATAC", "Misar-E13", atac_feature, mis13_atac_count, mis13_barc)
mis15_rna = SingleData("GeneExpr", "Misar-E15", rna_feature, mis15_rna_count, mis15_barc)
mis15_atac = SingleData("ATAC", "Misar-E15", atac_feature, mis15_atac_count, mis15_barc)
mis18_rna = SingleData("GeneExpr", "Misar-E18", rna_feature, mis18_rna_count, mis18_barc)
mis18_atac = SingleData("ATAC", "Misar-E18", atac_feature, mis18_atac_count, mis18_barc)

ste12_rna = SingleData("GeneExpr", "Stereo-E12", rna_feature, ste12_rna_count, ste12_barc)
ste14_rna = SingleData("GeneExpr", "Stereo-E14", rna_feature, ste14_rna_count, ste14_barc)
ste16_rna = SingleData("GeneExpr", "Stereo-E16", rna_feature, ste16_rna_count, ste16_barc)

In [12]:
multi_dt = MultiomicDataset.from_singledata(
    mis13_rna, mis13_atac, mis15_rna, mis15_atac, mis18_rna, mis18_atac, ste12_rna, ste14_rna, ste16_rna)
print(multi_dt)

A MultiomicDataset object with 2 omics:
- GeneExpr: 10000 features, 50334 cells, 6 batches.
- ATAC: 100000 features, 5855 cells, 3 batches.
Joint cells:
- GeneExpr, ATAC: 5855 cells.


### Training

In [ ]:
model = Cobolt(dataset=multi_dt, lr=0.001, n_latent=10, batch_size=128) 
model.train(num_epochs=100)

In [ ]:
plt.plot(model.history['loss'])

In [ ]:
model.calc_all_latent()

latent = model.get_all_latent()

In [13]:
if False:
    latent_barcode = np.array([_.split('~')[1] for _ in latent[1]])
    df_latent = pd.DataFrame(latent[0], index=latent_barcode)
    df_latent.to_csv(f'{out_dir}/res.csv')

else:
    df_latent = pd.read_csv(f'{out_dir}/res.csv', index_col=0)
    df_latent.index = df_latent.index.astype('str')

In [14]:
ad_mis13_rna.obsm['X_emb'] = df_latent.loc[ad_mis13_rna.obs_names, :].values
ad_mis15_rna.obsm['X_emb'] = df_latent.loc[ad_mis15_rna.obs_names, :].values
ad_mis18_rna.obsm['X_emb'] = df_latent.loc[ad_mis18_rna.obs_names, :].values

ad_ste12_rna.obsm['X_emb'] = df_latent.loc[ad_ste12_rna.obs_names, :].values
ad_ste14_rna.obsm['X_emb'] = df_latent.loc[ad_ste14_rna.obs_names, :].values
ad_ste16_rna.obsm['X_emb'] = df_latent.loc[ad_ste16_rna.obs_names, :].values

In [15]:
ad_mosaic = sc.concat([ad_mis13_rna, ad_mis15_rna, ad_mis18_rna, ad_ste12_rna, ad_ste14_rna, ad_ste16_rna])
# ad_mosaic = get_umap(ad_mosaic, ['X_emb'])

In [16]:
# # plot
# wrap_warn_plot(ad_mosaic, 'X_emb_umap', ['Sample'])

In [18]:
anal_dir = f'../../../results/embeddings/Cobolt/Misar-Stereo'
os.makedirs(anal_dir, exist_ok=True)

df = pd.DataFrame(ad_mosaic.obsm['X_emb'], index=ad_mosaic.obs_names)
df.to_csv(join(anal_dir, 'df_emb.csv'))